<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/2_06_Medallion_Architecture_Data_Lakehouse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.06 Medallion Architecture / Data Lakehouse
This notebook creates a simple medallion architecture / data lakehouse approach within Colab.

We will relax some of the previous rigour to make this a more natural Notebook progression (e.g. no `utilities.py` file and so on).

We'll start by creating directories on the machine where each of our layers will sit on:


In [1]:
import pandas as pd
import numpy as np
import os
import json
import duckdb

# Create our "Data Lake" directories
os.makedirs("datalake/bronze", exist_ok=True)
os.makedirs("datalake/silver", exist_ok=True)
os.makedirs("datalake/gold", exist_ok=True)

Next we will generate some artificial data to represent our raw data for the bronze layer.

In [2]:
# Generate "Messy" Raw Data
def generate_bronze_data():
    np.random.seed(42)
    data = []

    # We simulate 1000 raw events with some errors
    for i in range(1000):
        event = {
            "transaction_id": str(i) if np.random.rand() > 0.05 else None, # 5% missing IDs
            "user_id": np.random.randint(100, 110), # Only 10 users
            "amount": np.random.choice([10.5, 20.0, "50.0", -5.0, None]), # Mixed types & bad data
            "timestamp": "2024-02-10T10:00:00" if i % 2 == 0 else "2024-02-10 10:01:00", # Inconsistent dates
            "raw_metadata": "{\"source\": \"mobile\"}" # Nested JSON string
        }
        # Duplicate some rows to simulate ingestion errors
        if i % 50 == 0: # if this is the 50th row
            data.append(event) # duplicate it a second time
        data.append(event)

    return data

raw_data = generate_bronze_data()


Now we will create a simple function to load data into the bronze layer:

In [3]:
def process_bronze(data, output_path):
    # We effectively just dump the raw data into the bronze layer
    with open(output_path, "w") as f: # open the output file
        for entry in data: # loop through the data
            f.write(json.dumps(entry) + "\n") # write the data to output

    print(f"BRONZE LAYER: Generated {len(data)} raw records at {output_path}")

process_bronze(raw_data, "/content/datalake/bronze/raw_transactions.json")

BRONZE LAYER: Generated 1020 raw records at /content/datalake/bronze/raw_transactions.json


### Bronze to Silver (Cleaning & Validation)
Goal: Clean the data, enforce types, and deduplicate. Save as Parquet.

Why Parquet? It compresses data and stores schema information, making it much faster to read than JSON. You can read more [here](https://www.databricks.com/blog/what-is-parquet).

In [4]:
def process_silver(input_path, output_path):
    print("\nPROCESSING: Bronze -> Silver")

    # 1. Read Raw Data
    df = pd.read_json(input_path, lines=True)

    # 2. Data Cleaning Rules
    # Drop duplicates
    df = df.drop_duplicates()

    # Fix 'amount': Convert to numbers, coerce errors (strings) to NaN, fill NaNs with 0
    df["amount"] = pd.to_numeric(df["amount"], errors='coerce').fillna(0)

    # Filter out negative amounts (a Business Rule)
    df = df[df["amount"] >= 0]

    # Fix Timestamp format
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # 3. Enforce Schema & Save as Parquet
    # This ensures that downstream users don't have to guess data types.
    df.to_parquet(output_path, index=False)

    print(f"SILVER LAYER: Saved {len(df)} clean records to {output_path}")
    print(df.head(3)) # show top three records

process_silver("/content/datalake/bronze/raw_transactions.json", "/content/datalake/silver/clean_transactions.parquet")


PROCESSING: Bronze -> Silver
SILVER LAYER: Saved 783 clean records to /content/datalake/silver/clean_transactions.parquet
   transaction_id  user_id  amount           timestamp          raw_metadata
0             0.0      107     0.0 2024-02-10 10:00:00  {"source": "mobile"}
2             1.0      109    50.0 2024-02-10 10:01:00  {"source": "mobile"}
3             2.0      107     0.0 2024-02-10 10:00:00  {"source": "mobile"}


### Silver to Gold (Aggregation & Business Logic)
Goal: Create a business-level aggregate table (e.g., "Daily Sales per User").

This layer is highly curated. It is usually "read-optimised" for dashboards or engineered features for data science models.

In [5]:
def process_gold(input_path, output_path):
    print("\nPROCESSING: Silver -> Gold")

    # 1. Read Silver Data (Fast Parquet Read)
    df = pd.read_parquet(input_path)

    # 2. Aggregation (The "Business Logic")
    # aggregate the data by spend, transaction count and last seen
    gold_df = df.groupby("user_id").agg(
        total_spent=("amount", "sum"),
        transaction_count=("transaction_id", "count"),
        last_seen=("timestamp", "max")
    ).reset_index()

    # 3. Save to Gold as a parquet file
    gold_df.to_parquet(output_path, index=False)

    print(f"GOLD LAYER: Saved {len(gold_df)} summary records to {output_path}")
    print(gold_df.head(3))

process_gold("/content/datalake/silver/clean_transactions.parquet", "/content/datalake/gold/user_daily_summary.parquet")


PROCESSING: Silver -> Gold
GOLD LAYER: Saved 10 summary records to /content/datalake/gold/user_daily_summary.parquet
   user_id  total_spent  transaction_count           last_seen
0      100       1609.0                 71 2024-02-10 10:01:00
1      101       1405.5                 61 2024-02-10 10:01:00
2      102       1688.0                 78 2024-02-10 10:01:00


### Data Lakehouse
Lastly, we will build a data lakehouse using [DuckDB](https://motherduck.com/). Note, this is a virtual database that does not actually store the data. Instead we infer the schema, and return data, from the parquet files we saved in the Gold layer.

In [7]:
print("\nANALYST QUERY (DuckDB Lakehouse)")

# We use DuckDB to query the file directly.
# This mimics a "Serverless SQL" experience.

query = """
    SELECT
        user_id,
        total_spent
    FROM 'datalake/gold/user_daily_summary.parquet'
    ORDER BY total_spent DESC
    LIMIT 5;
"""

con = duckdb.connect(database=':memory:')
con.sql(query).show()


ANALYST QUERY (DuckDB Lakehouse)
┌─────────┬─────────────┐
│ user_id │ total_spent │
│  int64  │   double    │
├─────────┼─────────────┤
│     107 │      2082.0 │
│     103 │      1701.5 │
│     102 │      1688.0 │
│     109 │      1661.5 │
│     100 │      1609.0 │
└─────────┴─────────────┘



As we can see, this looks, feels and works like a normal data warehouse or RDBMs just its sitting directly on top of our data files.

## QUESTIONS


1.   Why not just query the Bronze file directly?
2.   Why do we need a Gold layer? Why not just aggregate Silver every time?
3. If I find a bug in the Silver cleaning logic, what do I do?

